# CS2235: Data Science Capstone Project
## Fintech: Cryptocurrency Risk Profiling via Unsupervised Clustering
**RV University | School of Computer Science & Engineering**  
*Programme:* B.Sc (Hons.) Data Science, Semester III  
*Instructors:* Prof. Muralidhar Billa, Prof. Thirukumaran Subbiramani  
*Researcher / Team Lead:* Thanisha S Rao  

---

### Problem Statement
> **Grouping cryptocurrencies into similar risk categories based on how much their prices change (volatility, drawdowns, tail risk) and how much they are traded (trading volume, turnover, market depth).**

---

### Project Workflow & Roadmap
1. **Phase 1: Project Kickoff & Setup** (Problem Formulation, Data Source Identification)
2. **Phase 2: Data Acquisition & Preprocessing** (Historical 2-Year Daily OHLCV Ingestion for 28 Assets)
3. **Phase 3: ETL & Mathematical Risk Feature Engineering** (Volatility, MDD, 95% VaR, CVaR, Sortino, Volume Velocity, Amihud)
4. **Phase 4: Machine Learning Clustering Implementation** (K-Means++, Agglomerative Hierarchical, GMM, DBSCAN)
5. **Phase 5: Cluster Validation, Diagnostics & 2D/3D PCA Decomposition** (Elbow Curves, Silhouette, Davies-Bouldin, Latent Space)
6. **Phase 6: Empirical Risk Profiling & Financial Insights** (Tier Archetypes, Portfolio Construction)


## 1. Environment Setup & Library Ingestion

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

print("All dependencies successfully imported.")


## 2. Phase 2: Data Collection & Integrity Verification
We inspect the ingested daily historical OHLCV dataset covering 28 diverse cryptocurrencies across 2 full years (731 daily trading periods per asset).

In [ ]:
# Load raw historical time-series data
raw_df = pd.read_csv('../data/raw/crypto_historical_raw.csv')
raw_df['Date'] = pd.to_datetime(raw_df['Date'])

print(f"Total time-series records: {len(raw_df):,}")
print(f"Total unique digital assets: {raw_df['Symbol'].nunique()}")
print(f"Time span: {raw_df['Date'].min().strftime('%Y-%m-%d')} to {raw_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Missing values count per column:\n{raw_df.isnull().sum()}")

raw_df.head()


## 3. Phase 3: ETL & Mathematical Risk Feature Engineering
We extract cross-sectional risk and liquidity metrics for each asset:
- **Price Volatility:** Annualized Volatility ($\sigma_{ann} = \sigma_{daily} \times \sqrt{365}$), High-Low Daily Spread
- **Downside Risk:** Maximum Drawdown (MDD), 95% Historical Value at Risk (VaR), Conditional VaR (CVaR / Expected Shortfall), Downside Deviation
- **Trading Liquidity:** Log10 Mean Daily Dollar Volume, Volume Volatility (CV), Amihud Illiquidity Ratio Proxy, Volatility-to-Volume Ratio

In [ ]:
# Load processed quantitative risk features
features_df = pd.read_csv('../data/processed/crypto_risk_features.csv')
print(f"Extracted feature shape: {features_df.shape}")

display_cols = ['Symbol', 'Name', 'Asset_Category', 'Annualized_Volatility_%', 
                'Max_Drawdown_%', 'VaR_95_%', 'CVaR_95_%', 'Log_Avg_Volume_USD', 'Amihud_Illiquidity']
features_df[display_cols].head(10)


### Feature Correlation Matrix

In [ ]:
feature_cols = [
    "Annualized_Volatility_%", "Max_Drawdown_%", "VaR_95_%", "CVaR_95_%",
    "Downside_Deviation_%", "Mean_Daily_Spread_%", "Log_Avg_Volume_USD",
    "Volume_Volatility_CV", "Amihud_Illiquidity", "Vol_to_Volume_Ratio"
]

plt.figure(figsize=(10, 8), dpi=150)
sns.heatmap(features_df[feature_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Quantitative Risk & Liquidity Metrics", fontsize=13, fontweight="bold")
plt.show()


## 4. Phase 4: Machine Learning Clustering Pipeline
We standardize the features using Z-score standardization ($Z = \frac{X - \mu}{\sigma}$) and evaluate optimal clusters ($k=2 \dots 7$) using Inertia, Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Index.

In [ ]:
CLUSTER_FEATURES = [
    "Annualized_Volatility_%",
    "Max_Drawdown_%",
    "VaR_95_%",
    "CVaR_95_%",
    "Mean_Daily_Spread_%",
    "Log_Avg_Volume_USD",
    "Volume_Volatility_CV",
    "Vol_to_Volume_Ratio"
]

X = features_df[CLUSTER_FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Evaluate K across 2 to 7
k_range = range(2, 8)
inertias, silhouettes, db_indices, ch_indices = [], [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=25, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    db_indices.append(davies_bouldin_score(X_scaled, labels))
    ch_indices.append(calinski_harabasz_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
ax1.plot(list(k_range), inertias, marker='o', linewidth=2, color='#1f77b4')
ax1.axvline(4, color='red', linestyle='--', label='Optimal k=4')
ax1.set_title("Elbow Method (Inertia vs. k)", fontweight="bold")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Within-Cluster Sum of Squares (Inertia)")
ax1.legend()

ax2.plot(list(k_range), silhouettes, marker='s', linewidth=2, color='#2ca02c')
ax2.axvline(4, color='red', linestyle='--', label='Optimal k=4')
ax2.set_title("Silhouette Score vs. k (Peak Separation)", fontweight="bold")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.legend()
plt.show()


## 5. Phase 5: Dimensionality Reduction (PCA & t-SNE) and Hierarchical Dendrogram
We project the 8-dimensional risk space into 2 principal components (which capture **93.78%** of the total variance) and construct an Agglomerative Hierarchical tree with Ward linkage.

In [ ]:
# Load clustered results
clustered_df = pd.read_csv('../data/processed/crypto_clustered_results.csv')

# 2D PCA Visualization
plt.figure(figsize=(11, 7), dpi=150)
tier_colors = {
    "Ultra-Low Risk": "#2ca02c",
    "Low-Moderate Risk": "#1f77b4",
    "Moderate-High Risk": "#ff7f0e",
    "Extreme Risk": "#d62728"
}

for tier, col in tier_colors.items():
    sub = clustered_df[clustered_df["Risk_Tier"] == tier]
    plt.scatter(sub["PCA1"], sub["PCA2"], label=tier, color=col, s=140, edgecolors="black", linewidths=0.8)

for _, row in clustered_df.iterrows():
    plt.annotate(row["Symbol"], (row["PCA1"], row["PCA2"]), xytext=(4, 4), textcoords="offset points", fontsize=8, fontweight="bold")

plt.title("2D Principal Component Analysis (PCA) Risk Clusters (93.78% Variance)", fontsize=13, fontweight="bold")
plt.xlabel("PCA Component 1 (Volatility & Downside Tail Risk)", fontweight="bold")
plt.ylabel("PCA Component 2 (Trading Liquidity & Turnover Depth)", fontweight="bold")
plt.legend(title="Risk Tier", frameon=True, facecolor="white")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


### Hierarchical Clustering Dendrogram (Ward Linkage)

In [ ]:
linkage_matrix = linkage(X_scaled, method="ward")
plt.figure(figsize=(13, 6), dpi=150)
dendrogram(linkage_matrix, labels=clustered_df["Symbol"].values, leaf_rotation=90, leaf_font_size=9)
plt.title("Agglomerative Hierarchical Clustering Dendrogram", fontsize=13, fontweight="bold")
plt.xlabel("Cryptocurrency Asset", fontweight="bold")
plt.ylabel("Euclidean Distance (Dissimilarity)", fontweight="bold")
plt.show()


## 6. Phase 6: Volatility vs. Volume Liquidity Frontier (The Core Problem Statement)
We graph how much cryptocurrency prices change (Annualized Volatility %) against how much they are traded (Log10 Average Daily Volume USD).

In [ ]:
plt.figure(figsize=(11, 6), dpi=150)

for tier, col in tier_colors.items():
    sub = clustered_df[clustered_df["Risk_Tier"] == tier]
    plt.scatter(sub["Log_Avg_Volume_USD"], sub["Annualized_Volatility_%"], 
                label=tier, color=col, s=140, edgecolors="black", linewidths=0.8)

for _, row in clustered_df.iterrows():
    plt.annotate(row["Symbol"], (row["Log_Avg_Volume_USD"], row["Annualized_Volatility_%"]), 
                 xytext=(5, 4), textcoords="offset points", fontsize=8, fontweight="bold")

plt.title("Core Problem Statement: Volatility vs. Trading Volume Depth", fontsize=13, fontweight="bold")
plt.xlabel("Log10 Average Daily Trading Volume (USD) [Liquidity Proxy]", fontweight="bold")
plt.ylabel("Annualized Price Volatility (%) [Price Change Proxy]", fontweight="bold")
plt.legend(title="Assigned Risk Tier", frameon=True, facecolor="white")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 7. Cluster Risk Profile Breakdown & Insights
Below is the empirical synthesis of the 4 discovered risk tiers:

In [ ]:
summary_table = clustered_df.groupby("Risk_Tier").agg({
    "Symbol": ["count", lambda s: ", ".join(s)],
    "Annualized_Volatility_%": "mean",
    "Max_Drawdown_%": "mean",
    "VaR_95_%": "mean",
    "Avg_Daily_Volume_USD": lambda v: f"${v.mean()/1e6:.1f}M"
})
summary_table.columns = ["Asset Count", "Cryptocurrencies", "Mean Volatility (%)", "Mean MDD (%)", "Mean 95% VaR (%)", "Mean Daily Volume"]
summary_table


### Conclusions & Practical Takeaways:
1. **Tier 1 (Ultra-Low Risk / Pegged Stability):** USDT, USDC, DAI show near-zero volatility (<0.7%) and minimal drawdowns (<0.5%), serving as stable baseline cash equivalents.
2. **Tier 2 (Low-to-Moderate Risk / Blue-Chips):** BTC, ETH, BNB benefit from deep institutional liquidity (>\$2B daily volume) and moderate relative volatility (44% - 69%), acting as foundational portfolio anchors.
3. **Tier 3 (Moderate-to-High Risk / Utility & Layer 1s):** SOL, XRP, ADA, AVAX, DOT, LINK, etc. exhibit active volume with 70% - 95% volatility, ideal for tactical alpha generation with managed downside controls.
4. **Tier 4 (Extreme Speculative Risk / Meme Tokens & High-Beta):** BONK, WIF, FLOKI, SHIB, LDO, CRV show severe tail risk (>100% volatility, >85% drawdowns), requiring strict capital caps and stop-loss rules.
